# EODHD Bollinger — Colab
Upload the MK_BB_EODHD_Colab_Netlify.zip source package first. Add EODHD_API_TOKEN to Colab Secrets. This notebook does not publish to Netlify by itself.

In [ ]:
from google.colab import files, userdata
import os, zipfile
from pathlib import Path
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
root = Path('/content/mk_bb_eodhd')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        dest = (root/member.filename).resolve()
        if not dest.is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive path')
        if member.filename in ('config.json','universe.json') and dest.exists():
            print('Preserved:', member.filename)
            continue
        z.extract(member, root)
os.chdir(root)
os.environ['EODHD_API_TOKEN'] = userdata.get('EODHD_API_TOKEN')
%pip -q install -r requirements.txt


In [ ]:
!python -m unittest -v test_engine
!python bb_eodhd.py --init
!python bb_eodhd.py --catalog


## Symbol validation — v2.1
private/catalog.json contains real INDX and FOREX records. COMM is never queried. Check universe.json settings for unresolved indices. USD spot-metal pairs are verified when present in the catalog. Commodity date/value series are plotted at their native frequency via commodities.py; OHLCV cannot be produced, so stopped backtests stay disabled. Soybeans/Cocoa appear as unsupported slots.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'bb_eodhd.py'], check=True)
files.download('netlify_site.zip')


For daily automation, upload the .github/workflows/daily.yml file from the source package to a private GitHub repository. Define EODHD_API_TOKEN, NETLIFY_AUTH_TOKEN and NETLIFY_SITE_ID as Actions Secrets. After the first successful build and the access/license check, set the PUBLISH_APPROVED repository variable to true. See README.md.

## Full runner — upload, validate, test, build, download
Alternative to the individual cells above: this single cell does the whole flow. Do not run both.

In [ ]:
# ============================================================
# EODHD BOLLINGER v4.0 — COLAB RUNNER
# Uploads the current source ZIP, tests it, builds the site package.
# This cell does NOT publish to Netlify.
# ============================================================

import os
import sys
import json
import zipfile
import subprocess
from pathlib import Path

from google.colab import files, userdata
from IPython.display import display

ROOT = Path("/content/mk_bb_eodhd")
ROOT.mkdir(parents=True, exist_ok=True)

# Reset success state from a previous run.
BUILD_OK = False

# 1. Read the Colab secret; never print the key.
try:
    token = userdata.get("EODHD_API_TOKEN").strip()
except Exception:
    raise RuntimeError(
        "Add EODHD_API_TOKEN under the notebook's Secrets panel (left sidebar) "
        "and enable Notebook access. GitHub Secrets are not shared with Colab."
    ) from None

if not token:
    raise RuntimeError("EODHD_API_TOKEN is empty.")

os.environ["EODHD_API_TOKEN"] = token
del token

# 2. Select the current source package.
print("Select the current MK_BB_EODHD_Colab_Netlify.zip source package.")
uploaded = files.upload()

archives = [name for name in uploaded if name.lower().endswith(".zip")]

if len(archives) != 1:
    raise RuntimeError("Please upload exactly one source ZIP file.")

archive_path = Path(archives[0]).resolve()

required_files = {
    "bb_eodhd.py",
    "commodities.py",
    "netlify_setup.py",
    "requirements.txt",
    "test_engine.py",
    "config.json",
    "universe.json",
}

# 3. Validate the ZIP and extract it safely.
with zipfile.ZipFile(archive_path) as archive:
    missing = required_files - set(archive.namelist())

    if missing:
        raise RuntimeError(
            "This is not the current source package. Missing files: "
            + ", ".join(sorted(missing))
        )

    for member in archive.infolist():
        destination = (ROOT / member.filename).resolve()

        if not destination.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Unsafe file path inside the ZIP.")

    for member in archive.infolist():
        destination = ROOT / member.filename

        if (
            member.filename in {"config.json", "universe.json"}
            and destination.exists()
        ):
            print("Existing settings preserved:", member.filename)
            continue

        archive.extract(member, ROOT)

os.chdir(ROOT)

def run_step(title, arguments):
    print(f"\n{title}", flush=True)
    subprocess.run(
        [sys.executable, *arguments],
        cwd=ROOT,
        check=True,
    )

# 4. Dependencies and tests.
run_step(
    "1/4 — Installing Python dependencies",
    ["-m", "pip", "install", "-q", "-r", "requirements.txt"],
)

run_step(
    "2/4 — Running validation tests",
    ["-m", "unittest", "-v", "test_engine"],
)

run_step(
    "3/4 — Checking configuration",
    ["bb_eodhd.py", "--init"],
)

# 5. Fetch real data and build the site.
try:
    run_step(
        "4/4 — Fetching EODHD data and building the report",
        ["bb_eodhd.py"],
    )
except subprocess.CalledProcessError:
    audit_path = ROOT / "private" / "latest_audit.json"

    if audit_path.exists():
        import pandas as pd

        print("\nLast saved Data Audit:")
        display(
            pd.DataFrame(
                json.loads(audit_path.read_text(encoding="utf-8"))
            )
        )

    raise RuntimeError(
        "This run did not complete. Share the error output above. "
        "The previously generated ZIP has not been replaced."
    ) from None

# 6. Verify the generated site package.
site_zip = ROOT / "netlify_site.zip"

if not site_zip.exists():
    raise RuntimeError("netlify_site.zip not found after the run.")

with zipfile.ZipFile(site_zip) as archive:
    if "index.html" not in archive.namelist():
        raise RuntimeError("index.html missing from the site package.")

    if "portfolio.json" not in archive.namelist():
        raise RuntimeError("portfolio.json missing from the site package (v4.0 build expected).")

    if archive.testzip() is not None:
        raise RuntimeError("Site ZIP integrity check failed.")

BUILD_OK = True

print("\nSUCCESS: netlify_site.zip with index.html created.")
print("This does not mean all 47 assets were accessible.")
print("Review missing or unsuitable series in the report's Data Audit section.")

files.download(str(site_zip))
